# From PDF to Intelligence: Contextual Chunking + Multimodal Embeddings

This notebook demonstrates two advanced VoyageAI capabilities on a real PDF document:

| Model | What it does |
|-------|-------------|
| `voyage-3-lite` | Fast text embeddings (baseline) |
| `voyage-context-3` | Embeddings that understand each chunk's role in the full document |
| `voyage-multimodal-3.5` | Shared embedding space for text AND images |

**The Story:** We take the "Attention Is All You Need" paper (the Transformer paper) and show how each generation of VoyageAI models unlocks new retrieval capability — from crude keyword-like text search all the way to searching charts and diagrams with natural language.

In [ ]:
!pip install -q voyageai pymupdf pillow numpy matplotlib seaborn requests python-dotenv

In [ ]:
import os, io, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import fitz  # PyMuPDF
import requests
from PIL import Image

# ── API Key Setup (Google Colab compatible) ──────────────────────────────────
try:
    from google.colab import userdata
    VOYAGE_API_KEY = userdata.get('VOYAGE_API_KEY')
    print("Loaded API key from Google Colab Secrets")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    VOYAGE_API_KEY = os.environ.get('VOYAGE_API_KEY')
    print("Loaded API key from .env file")
except Exception:
    VOYAGE_API_KEY = None
    print("Could not load API key from Colab Secrets")

assert VOYAGE_API_KEY, "VOYAGE_API_KEY not found — set it in Colab Secrets or .env"

import voyageai
vo = voyageai.Client(api_key=VOYAGE_API_KEY)
print("VoyageAI client ready.")

In [ ]:
# ── Download PDF ─────────────────────────────────────────────────────────────
PDF_URL = "https://arxiv.org/pdf/1706.03762"
PDF_PATH = "/tmp/attention_is_all_you_need.pdf"

if not os.path.exists(PDF_PATH):
    print(f"Downloading PDF from {PDF_URL} ...")
    r = requests.get(PDF_URL, stream=True, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    r.raise_for_status()
    with open(PDF_PATH, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Saved to {PDF_PATH} ({os.path.getsize(PDF_PATH)//1024} KB)")
else:
    print(f"PDF already cached at {PDF_PATH}")

# Verify it's a valid PDF
doc = fitz.open(PDF_PATH)
print(f"PDF has {len(doc)} pages.")
doc.close()

In [ ]:
# ── Helper Functions ──────────────────────────────────────────────────────────

def cosine(a, b):
    """Cosine similarity between two vectors."""
    a, b = np.array(a, dtype=float), np.array(b, dtype=float)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        raise ValueError("Cannot compute cosine similarity with a zero vector")
    return float(np.dot(a, b) / (na * nb))

def standard_chunk(text, size=500, overlap=50):
    """Split text into fixed-size overlapping character chunks."""
    if overlap >= size:
        raise ValueError(f"overlap ({overlap}) must be less than size ({size})")
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        chunks.append(text[start:end])
        start += size - overlap
    return [c for c in chunks if c.strip()]

def render_page_image(pdf_path, page_num, dpi=150):
    """Render a single PDF page as a PIL Image."""
    doc = fitz.open(pdf_path)
    try:
        page = doc[page_num]
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        pix = page.get_pixmap(matrix=mat)
        return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    finally:
        doc.close()

def embed_in_batches(texts, model, input_type, batch_size=50):
    """Embed a list of texts in batches to avoid token limits."""
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        result = vo.embed(batch, model=model, input_type=input_type)
        embeddings.extend(result.embeddings)
    return embeddings

print("Helper functions defined.")

---
## Part A — The Baseline: Standard Chunking + `voyage-3-lite`

We start simple: extract text from the PDF, split it into fixed-size chunks, embed with `voyage-3-lite`, and run semantic search.

**The limitation we'll expose:** Fixed-size chunks truncate mid-sentence and lack document context. A chunk about "multi-head attention" has no idea it's inside a section called "Model Architecture." This hurts retrieval quality for questions that span sections.

In [ ]:
# ── Extract text per page ────────────────────────────────────────────────────
doc = fitz.open(PDF_PATH)
page_texts = [page.get_text() for page in doc]
doc.close()

full_text = "
".join(page_texts)
print(f"Extracted {len(full_text):,} characters across {len(page_texts)} pages")
print(f"
Sample (chars 2000-2500):
{'─'*60}")
print(full_text[2000:2500])

In [ ]:
# ── Create standard chunks ────────────────────────────────────────────────────
all_chunks = standard_chunk(full_text, size=500, overlap=50)
print(f"Created {len(all_chunks)} chunks (size=500, overlap=50)")
print(f"
Chunk 10 preview:
{'─'*60}")
print(all_chunks[10])
print(f"
Chunk 11 preview:
{'─'*60}")
print(all_chunks[11])

In [ ]:
# ── Embed with voyage-3-lite ─────────────────────────────────────────────────
print("Embedding chunks with voyage-3-lite...")
standard_embeddings = embed_in_batches(all_chunks, model="voyage-3-lite", input_type="document")

print(f"Embedded {len(standard_embeddings)} chunks")
print(f"Embedding dimension: {len(standard_embeddings[0])}")
assert len(standard_embeddings) == len(all_chunks), "Mismatch between chunks and embeddings"

# Guard: verify no zero-vector embeddings (would cause ValueError in cosine())
zero_mask = [np.linalg.norm(e) == 0 for e in standard_embeddings]
assert not any(zero_mask), f"Zero-vector embeddings at indices: {[i for i,z in enumerate(zero_mask) if z]}"
print("No zero-vector embeddings detected.")

In [ ]:
# ── Retrieval with voyage-3-lite ─────────────────────────────────────────────
def retrieve_standard(query, top_k=5):
    q_emb = vo.embed([query], model="voyage-3-lite", input_type="query").embeddings[0]
    scores = [(cosine(q_emb, emb), i) for i, emb in enumerate(standard_embeddings)]
    scores.sort(reverse=True)
    return [(score, all_chunks[idx], idx) for score, idx in scores[:top_k]]

# Three test queries: text-in-body, text-near-chart, text-about-diagram
QUERIES = [
    "What is the attention mechanism and how does it work?",
    "How does the model perform on WMT translation benchmarks?",
    "What are the encoder and decoder components of the Transformer?",
]

# standard_results: {query_str: [(score: float, chunk: str, chunk_idx: int), ...]}
standard_results = {}
for q in QUERIES:
    standard_results[q] = retrieve_standard(q, top_k=5)

print("Standard Chunking Results")
print("=" * 70)
for q in QUERIES:
    print(f"\n▶ Query: {q}")
    for rank, (score, chunk, idx) in enumerate(standard_results[q][:3], 1):
        preview = chunk.replace("\n", " ")[:120]
        print(f"  [{rank}] score={score:.4f} chunk#{idx:03d}: {preview}…")

### Observations

Notice how some high-scoring chunks are fragments that lack context:
- A chunk might start mid-sentence: *"...the encoder maps an input sequence of symbol..."*
- Chunks near figures contain only caption text — the visual information in the figure is invisible to text-only retrieval
- For the benchmark query, the top results may miss the actual numbers because they're in a table (tables often don't extract well as plain text)

**In Part B, we give the model the full document so it can understand what each chunk *means* in context.**

---
## Part B — Context-Aware Chunking with `voyage-context-3`

`voyage-context-3` receives **all chunks of a document together**, allowing it to encode each chunk with awareness of where it sits in the document. A chunk about "scaled dot-product attention" now knows it's inside "Section 3.2: Attention" of a paper about Transformers.

**API:** `POST /v1/contextualizedembeddings` — pass all chunks as one inner list; the model processes them jointly.

In [ ]:
# ── Embed with voyage-context-3 ───────────────────────────────────────────────
def contextualized_embed(chunks, model="voyage-context-3"):
    """
    Call the contextualizedembeddings REST endpoint.
    All chunks are passed as a single inner list so the model
    can encode each chunk with full document context.
    """
    headers = {
        "Authorization": f"Bearer {VOYAGE_API_KEY}",
        "content-type": "application/json",
    }
    payload = {
        "inputs": [chunks],       # one document = one inner list of all its chunks
        "input_type": "document",
        "model": model,
    }
    resp = requests.post(
        "https://api.voyageai.com/v1/contextualizedembeddings",
        headers=headers,
        json=payload,
        timeout=120,
    )
    try:
        resp.raise_for_status()
    except requests.HTTPError as e:
        print(f"API error body: {resp.text[:500]}")
        raise
    data = resp.json()
    return data["results"][0]["embeddings"]

print(f"Sending {len(all_chunks)} chunks to voyage-context-3...")
contextual_embeddings = contextualized_embed(all_chunks)

print(f"Received {len(contextual_embeddings)} contextual embeddings")
print(f"Embedding dimension: {len(contextual_embeddings[0])}")
assert len(contextual_embeddings) == len(all_chunks)
zero_mask = [np.linalg.norm(e) == 0 for e in contextual_embeddings]
assert not any(zero_mask), f"Zero-vector embeddings at indices: {[i for i,z in enumerate(zero_mask) if z]}"
print("No zero-vector embeddings detected.")

In [ ]:
# ── Retrieval with voyage-context-3 ──────────────────────────────────────────
def retrieve_contextual(query, top_k=5):
    # Query side: embed with voyage-3-lite (same as Part A for fair comparison)
    q_emb = vo.embed([query], model="voyage-3-lite", input_type="query").embeddings[0]
    scores = [(cosine(q_emb, emb), i) for i, emb in enumerate(contextual_embeddings)]
    scores.sort(reverse=True)
    return [(score, all_chunks[idx], idx) for score, idx in scores[:top_k]]

# contextual_results: {query_str: [(score: float, chunk: str, chunk_idx: int), ...]}
contextual_results = {}
for q in QUERIES:
    contextual_results[q] = retrieve_contextual(q, top_k=5)

print("Contextual Chunking Results")
print("=" * 70)
for q in QUERIES:
    print(f"\n▶ Query: {q}")
    for rank, (score, chunk, idx) in enumerate(contextual_results[q][:3], 1):
        preview = chunk.replace("\n", " ")[:120]
        print(f"  [{rank}] score={score:.4f} chunk#{idx:03d}: {preview}…")

In [ ]:
# ── Side-by-Side Comparison ───────────────────────────────────────────────────
for q in QUERIES:
    print(f"\n{'='*72}")
    print(f"Query: {q}")
    print(f"{'Standard (voyage-3-lite)':<36} {'Contextual (voyage-context-3)':<36}")
    print("─" * 72)
    for rank in range(3):
        s_score, s_chunk, s_idx = standard_results[q][rank]
        c_score, c_chunk, c_idx = contextual_results[q][rank]
        s_preview = s_chunk.replace("\n", " ")[:55]
        c_preview = c_chunk.replace("\n", " ")[:55]
        print(f"[{rank+1}] {s_score:.4f} #{s_idx:03d} {s_preview:<55}")
        print(f"    {c_score:.4f} #{c_idx:03d} {c_preview:<55}")
        print()

### Key Insight

`voyage-context-3` typically surfaces better chunks for queries about **specific technical concepts** because the model sees:
- Where the chunk appears in the document structure (intro vs. methods vs. results)
- What sections come before and after it
- Whether a chunk is a definition, an experiment result, or a transition

For the benchmark query, contextual embeddings often correctly elevate chunks containing numerical results because they "know" those chunks are in an experimental section — not just floating text.

**What's still missing?** Both approaches are *blind to the figures and tables*. The architecture diagram, the scaled dot-product attention figure, and the performance chart are invisible to text-only models. Part C fixes this.